# Grover core — the shared building blocks

This notebook contains **all the functions that the other notebooks use** to build and analyse Grover circuits.
Other notebooks load it with

```python
%run grover_core.ipynb
```

which runs every cell here and makes the functions available, so the circuit is defined in exactly one place.
This notebook therefore contains only definitions (nothing is simulated here).

In [33]:
import math

from qiskit import QuantumCircuit

In [34]:
def _check_n(n):
    if not isinstance(n, int) or n < 2:
        raise ValueError(
            f"n must be an integer >= 2 (got n={n!r}). "
        )


def _as_list(marked):
    if isinstance(marked, str):
        marked = [marked]
    return list(dict.fromkeys(marked))


def _check_marked(n, marked):
    states = _as_list(marked)
    if not states:
        raise ValueError("At least one marked state is required.")
    for s in states:
        if len(s) != n or set(s) - {"0", "1"}:
            raise ValueError(
                f"Marked state {s!r} is not a {n}-character bitstring of 0s and 1s."
            )
    return states

In [35]:
def oracle(n, marked):
    _check_n(n)
    states = _check_marked(n, marked)

    qc = QuantumCircuit(n, name="Oracle")
    for state in states:
        zeros = [i for i, b in enumerate(reversed(state)) if b == "0"]
        if zeros:
            qc.x(zeros)              
        qc.h(n - 1)                 
        qc.mcx(list(range(n - 1)), n - 1)
        qc.h(n - 1)
        if zeros:
            qc.x(zeros) 
    return qc

In [36]:
def diffuser(n):
    _check_n(n)
    qc = QuantumCircuit(n, name="Diffuser")
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc

In [37]:
def _theta(n, k):
    _check_n(n)
    N = 2 ** n
    if not 1 <= k <= N:
        raise ValueError(f"k must be between 1 and N={N} (got k={k}).")
    return math.asin(math.sqrt(k / N))


def optimal_iterations(n, k=1):
    theta = _theta(n, k)
    return round(math.pi / (4 * theta) - 0.5)


def theoretical_success(n, r, k=1):
    theta = _theta(n, k)
    return math.sin((2 * r + 1) * theta) ** 2

In [38]:
def grover(n, marked, iterations=None):
    _check_n(n)
    states = _check_marked(n, marked)
    if iterations is None:
        iterations = optimal_iterations(n, len(states))
    if iterations < 0:
        raise ValueError(f"iterations must be >= 0 (got {iterations}).")

    orc = oracle(n, states)
    dif = diffuser(n)

    qc = QuantumCircuit(n)
    qc.h(range(n))
    for _ in range(iterations):
        qc.compose(orc, inplace=True)
        qc.compose(dif, inplace=True)
    qc.measure_all()
    return qc

In [39]:
def success_rate(counts, marked, shots):
    hits = sum(counts.get(s, 0) for s in _as_list(marked))
    return hits / shots

In [40]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

NOISE_BASIS_GATES = ["rz", "sx", "x", "cx"]


def depolarizing_noise_model(p2, p1=None):
    if p1 is None:
        p1 = p2 / 10
    if not (0 <= p1 <= 1 and 0 <= p2 <= 1):
        raise ValueError(f"Error probabilities must be between 0 and 1 (got p1={p1}, p2={p2}).")
    nm = NoiseModel(basis_gates=NOISE_BASIS_GATES)
    if p1 > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ["sx", "x"])
    if p2 > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ["cx"])
    return nm